In [1]:
import pygpc
import numpy as np
from collections import OrderedDict
import matplotlib.pyplot as plt
from pygpc.AbstractModel import AbstractModel
import os
import time
import h5py
import warnings
import matplotlib.cbook
from collections import OrderedDict
from mpl_toolkits.mplot3d import Axes3D
from IPython import display
import inspect
import scipy
import subprocess
import xarray as xr

In [2]:
### Setting up a model

class APCEMM(AbstractModel):

    def __init__(self):
        self.fname = inspect.getfile(inspect.currentframe())

    def validate(self):
        pass

    def simulate(self, process_id=None, matlab_engine=None):

        if self.p["x1"] is not np.ndarray:
            self.p["x1"] = np.array(self.p["x1"])

        if self.p["x2"] is not np.ndarray:
            self.p["x2"] = np.array(self.p["x2"])

        # RUN APCEMM WITH SAMPLES
        call_APCEMM(self.p["x1"].flatten()[0],self.p["x2"].flatten()[0])


        print("finished apcemm run")
        # Wait until APCEMM has run to continue
        job_ID = os.environ["job_ID"]
        path2output = '/home/chinahg/GCresearch/APCEMM/rundirs/SampleRunDir/slurm/'+str(job_ID)+'/ts_aerosol_case0_0100.nc' # THIS IS AT A SPECIFIC TIME, NEED TO FIGURE OUT WHICH SLICE TO LOOK AT IN THE TIME SERIES

        y = vod_APCEMM(path2output)

        if type(y) is not np.ndarray:
            y = np.array([y])

        y_out = y[:, np.newaxis]

        return y_out

def call_APCEMM(x1, x2):
    subprocess.call(['bash', 'uncertainty.sh', 'x1', 'x2']) #call bash script and pass arguments x1, x2
    p1 = subprocess.run("sbatch /home/chinahg/GCresearch/APCEMM/rundirs/SampleRunDir/uncertainty.sh", shell = True)
    p1.wait()

def vod_APCEMM(path2output):
    # Open result file and take mode of VOD parameter
    while os.path.exists(path2output) == False:
        counter = counter + 1
        time.sleep(60*15) # Wait 15 minutes before looking again
    
    print("File found after ", counter*15, " minutes.")

    apce_data = read_apcemm_data('path2output')
    ds_t = apce_data.ds_t

    all_vod = ds_t[0]["Vertical optical depth"]
    vod_mode = scipy.stats.mode(all_vod)
    
    return vod_mode

#Functions that will be used for postprocessing
class apce_data_struct:
    def __init__(self, t, ds_t, icemass, h2omass, numparts):
        self.t = t
        self.ds_t = ds_t
        self.icemass = icemass
        self.h2omass = h2omass
        self.numparts = numparts
    
def read_apcemm_data(directory):
    t_mins = []
    ds_t = []
    ice_mass = []
    total_h2o_mass = []
    num_particles = []

    for file in sorted(os.listdir(directory)):
        if(file.startswith('ts_aerosol') and file.endswith('.nc')):
            file_path = os.path.join(directory,file)
            ds = xr.open_dataset(file_path, engine = "netcdf4", decode_times = False)
            ds_t.append(ds)
            tokens = file_path.split('.')
            mins = int(tokens[-2][-2:])
            hrs = int(tokens[-2][-4:-2])
            t_mins.append(hrs*60 + mins)

            ice_mass.append(ds["Ice Mass"])
            num_particles.append(ds["Number Ice Particles"])
            dx = abs(ds["x"][-1] - ds["x"][0])/len(ds["x"])
            dy = abs(ds["y"][-1] - ds["y"][0])/len(ds["y"])
            
            h2o_mass = np.sum(ds["H2O"]) * 1e6 / 6.022e23 * 0.018 * dx*dy + ds["Ice Mass"]
            total_h2o_mass.append(h2o_mass.values)
    return apce_data_struct(t_mins, ds_t, ice_mass, total_h2o_mass, num_particles)

def removeLow(arr, cutoff = 1e-3):
    func = lambda x: (x > cutoff) * x
    vfunc = np.vectorize(func)
    return vfunc(arr)

###################################################################################################################


In [3]:
# Model
model = APCEMM()

# Problem
parameters = OrderedDict()
parameters["x1"] = pygpc.Norm(pdf_shape=[5,1]) #mean, std of soot EI
parameters["x2"] = pygpc.Norm(pdf_shape=[1,4]) #sulfur fuel content
# parameters["x3"] = pygpc.Norm(pdf_shape=[3,7]) # diffusion coefficient

# Create grid, random sampling of 200 datapoints
grid = pygpc.Random(parameters_random=parameters,
                    n_grid=200,
                    options={"seed": None})

problem = pygpc.Problem(model, parameters)

# gPC options
options = dict()
options["solver"] = "LarsLasso" # Least angle regression
options["fn_results"] = "tmp/mygpc" 

# define algorithm
algorithm = pygpc.RegAdaptive(problem=problem, options=options)

# Initialize gPC Session
session = pygpc.Session(algorithm=algorithm)

# run gPC session
session, coeffs, results = session.run()

Initializing gPC object...


Creating initial grid (<class 'pygpc.Grid.Random'>) with n_grid=2.0
Initializing gPC matrix...
Extending grid from 2 to 2 by 0 sampling points
Performing simulations 1 to 2
Submitted batch job 360525


bash: uncertainty.sh: No such file or directory


AttributeError: 'CompletedProcess' object has no attribute 'wait'

In [ ]:
mean = session.gpc[0].get_mean(coeffs)
print("Mean: {}".format(mean))

In [ ]:
std = session.gpc[0].get_std(coeffs)
print("Std: {}".format(std))

In [ ]:
pdf_x, pdf_y = session.gpc[0].get_pdf(coeffs, n_samples=1e4, output_idx=[0])
plt.plot(pdf_x, pdf_y)
plt.xlabel("$y$", fontsize=16)
plt.ylabel("$p(y)$", fontsize=16)